In [2]:
import numpy as np
import pandas as pd
import glob
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from collections import defaultdict
from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation

#Alle csv- Dateien aus der angegebenen Quelle werden verarbeitet
dataset_path = glob.glob("C:\\Users\\Benedikt Thissen\\OneDrive\\Dokumente\\1Studium\\Data Analysis\\Datenset\\*.csv")

#Die Werte der csvs werden in tweets gelistet
tweets = []
for data in tqdm(dataset_path, desc="Verarbeite CSV-Dateien"):
    tw = pd.read_csv(data)
    tweets.append(tw)
    
all_tweets = pd.concat(tweets, ignore_index=True)

#Für den Corpus werden alle Einträge aus der Spalte content verwendet
corpus = all_tweets["content"].dropna().tolist()

#Textbereinigung (V2 vocabulary auf *set* geändert, damit keine doppelte Einträge vorhanden sind)
reviews = []
vocabulary = set()
sentences = []
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

#Funktion, die prüft ob Buchstaben mehr als 3 mal hintereinander kommen
def is_valid_word(word):
    if re.search(r"(.)\1{3,}", word):
        return False
    return True

#Textbereinigung Sonderzeichen, stopwords, nicht valide entfernt 
for r in tqdm(corpus, desc="Verarbeite Tweets"):
    tokens = re.findall(r'\b\w+\b', r.lower())
    rev = [lemmatizer.lemmatize(w) for w in tokens if w and w.isalpha() and w not in stop_words and is_valid_word(w)]
    sentences.append(rev)
    vocabulary.update(rev)

len_vocabulary = len(vocabulary)
print(f"Anzahl Wörter: {len_vocabulary}")

  
#BoW-Methode 

joined_sentences = [" ".join(words) for words in sentences]
vector = CountVectorizer(max_features=1000)
bow_data = vector.fit_transform(tqdm(joined_sentences, desc="Erzeuge BoW-Vektoren"))
bow_df = pd.DataFrame(bow_data.toarray(), columns=vector.get_feature_names_out())


#TF-IDF-Matirx erstellen. Wöteranzahl reduziert
vectorizer = TfidfVectorizer(max_features=1000)
model = vectorizer.fit_transform(tqdm(joined_sentences, desc="TF-IDF Verarbeitung"))
tfidf_data = pd.DataFrame(model.toarray(), columns=vectorizer.get_feature_names_out())

#Dimensionsreduktion mit der LSA
topics_count = 5
lsa_model = TruncatedSVD(n_components=topics_count, algorithm="randomized", n_iter=10, random_state=42)
lsa = lsa_model.fit_transform(tfidf_data)

terms = vectorizer.get_feature_names_out()
print("\n\n--- LSA Themenverteilung ---")
for topic_idx, topic in enumerate(lsa_model.components_):
    print(f"\nTop 10 Begriffe für Thema {topic_idx + 1}:")
    top_terms_idx = topic.argsort()[:-11:-1]
    top_terms = [terms[i] for i in top_terms_idx]
    print(top_terms)

for i in range(3):
    l = lsa[i]
    print(f"\nDokument {i}:")
    for j, topic in enumerate(l):
        print(f"  Thema {j}: {topic*100:.2f}%")
        
#LDA-Analyse 

lda_model = LatentDirichletAllocation(
    n_components=5,            
    learning_method='online',
    random_state=42,
    max_iter=10
)
lda_top = lda_model.fit_transform(model)

# Topics anzeigen
print("\n\n--- LDA Themenverteilung ---")
for i in range(3):
    print(f"\nDokument {i + 1}:")
    for j, topic in enumerate(lda_top[i]):
        print(f"  → Thema {j}: {topic * 100:.2f}%")

# Top-Wörter pro Thema
feature_names = vectorizer.get_feature_names_out()
for topic_idx, topic in enumerate(lda_model.components_):
    print(f"\nTop-Wörter für Thema {topic_idx}:")
    top_words_idx = topic.argsort()[:-11:-1]
    top_words = [feature_names[i] for i in top_words_idx]
    print("  ", ", ".join(top_words))




Verarbeite Tweets: 100%|██████████| 1316605/1316605 [03:01<00:00, 7238.55it/s]


Anzahl Wörter: 524751


TF-IDF Verarbeitung: 100%|██████████| 1316605/1316605 [00:21<00:00, 60183.86it/s]




--- LSA Themenverteilung ---

Top 10 Begriffe für Thema 1:
['co', 'http', 'standwithukraine', 'ukraine', 'russia', 'troop', 'nato', 'russian', 'war', 'border']

Top 10 Begriffe für Thema 2:
['standwithukraine', 'http', 'co', 'nowar', 'stoprussianaggression', 'stopputin', 'stopputinnow', 'thank', 'ukraineunderattack', 'ukraina']

Top 10 Begriffe für Thema 3:
['russian', 'troop', 'http', 'co', 'border', 'near', 'ukrainian', 'via', 'belarus', 'invasion']

Top 10 Begriffe für Thema 4:
['http', 'co', 'war', 'russia', 'invade', 'stand', 'via', 'crisis', 'youtube', 'tension']

Top 10 Begriffe für Thema 5:
['invade', 'russia', 'troop', 'border', 'russian', 'going', 'biden', 'say', 'standwithukraine', 'china']

Dokument 0:
  Thema 0: 24.56%
  Thema 1: -6.23%
  Thema 2: 15.44%
  Thema 3: -1.44%
  Thema 4: 1.63%

Dokument 1:
  Thema 0: 16.75%
  Thema 1: -15.62%
  Thema 2: 26.52%
  Thema 3: -27.24%
  Thema 4: 5.06%

Dokument 2:
  Thema 0: 13.17%
  Thema 1: -12.68%
  Thema 2: 6.22%
  Thema 3: -14